In [2]:
# pip install biopython
import json
import re
from Bio import Entrez

In [3]:
# --- CONFIGURATION ---
DISEASE_LIST = ["COVID-19", "Monkeypox", "Measles", "Influenza", "Ebola"]
MAX_ABSTRACTS_PER_DISEASE = 50
EMAIL = "e.sagheb@gmail.com"  # NCBI requires this
OUTPUT_FILE = "raw_abstracts.json"

In [4]:
def get_year_from_article(article_data):
    """
    Safely extracts the publication year from the nested XML structure.
    Handles 'Year' fields and 'MedlineDate' fields (e.g., '2020 May-Jun').
    """
    try:
        # Navigate to PubDate
        journal_issue = article_data.get('Journal', {}).get('JournalIssue', {})
        pub_date = journal_issue.get('PubDate', {})
        
        # Priority 1: Standard 'Year' tag
        if 'Year' in pub_date:
            return str(pub_date['Year'])
            
        # Priority 2: 'MedlineDate' tag (often used for ranges like "2020 Oct-Dec")
        if 'MedlineDate' in pub_date:
            medline_date = str(pub_date['MedlineDate'])
            # Regex to grab the first 4-digit number
            match = re.search(r'\d{4}', medline_date)
            if match:
                return match.group(0)
                
    except Exception:
        pass
        
    return "Unknown"

In [5]:
def fetch_targeted_abstracts(disease_list, max_results_per_disease, email):
    Entrez.email = email
    all_abstracts = []
    
    for disease in disease_list:
        print(f"\n--- Processing Disease: {disease} ---")
        
        # 1. Strict Query Construction
        search_term = (
                f"({disease}[tiab] OR {disease}[mesh]) AND "
                f"("
                f"\"basic reproduction number\"[tiab] OR "
                f"\"effective reproduction number\"[tiab] OR "
                f"\"reproduction number\"[tiab] OR "
                f"\"time-varying reproduction number\"[tiab] OR "
                f"\"instantaneous reproduction number\"[tiab] OR "
                f"R0[tiab] OR Rt[tiab] OR \"R(t)\"[tiab]"
                f") "
                f"NOT ("
                f"\"RT-PCR\"[tiab] OR "
                f"\"reverse transcription\"[tiab] OR "
                f"\"real-time PCR\"[tiab] OR "
                f"\"real time PCR\"[tiab]"
                f")"
            )
        # 2. Search for IDs
        try:
            search_handle = Entrez.esearch(db="pubmed", term=search_term, retmax=max_results_per_disease, sort="relevance")
            search_results = Entrez.read(search_handle)
            search_handle.close()
        except Exception as e:
            print(f"Error searching for {disease}: {e}")
            continue
        
        id_list = search_results["IdList"]
        if not id_list:
            print(f"No papers found for {disease}.")
            continue
            
        print(f"Found {len(id_list)} candidates for {disease}. Downloading details...")
        
        # 3. Fetch Details
        try:
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            papers = Entrez.read(fetch_handle)
            fetch_handle.close()
        except Exception as e:
            print(f"Error fetching details for {disease}: {e}")
            continue
        
        count = 0
        for article in papers['PubmedArticle']:
            try:
                # Extract MedlineCitation
                citation = article.get('MedlineCitation', {})
                article_data = citation.get('Article', {})
                
                # Metadata Extraction
                pmid = str(citation.get('PMID', ''))
                title = article_data.get('ArticleTitle', '')
                year = get_year_from_article(article_data)
                
                # Abstract Extraction
                abstract_text = ""
                if 'Abstract' in article_data:
                    abstract_parts = article_data['Abstract']['AbstractText']
                    if isinstance(abstract_parts, list):
                        abstract_text = " ".join([str(part) for part in abstract_parts])
                    else:
                        abstract_text = str(abstract_parts)
                
                if abstract_text:
                    entry = {
                        "disease_query": disease,
                        "pmid": pmid,
                        "title": title,
                        "year": year,
                        # We combine Title + Abstract for the LLM to read
                        "text": f"Title: {title}\nAbstract: {abstract_text}"
                    }
                    all_abstracts.append(entry)
                    count += 1
                    
            except Exception:
                continue
        print(f"Successfully saved {count} abstracts for {disease}.")

    return all_abstracts

In [6]:

data = fetch_targeted_abstracts(DISEASE_LIST, MAX_ABSTRACTS_PER_DISEASE, EMAIL)
    



--- Processing Disease: COVID-19 ---
Found 50 candidates for COVID-19. Downloading details...
Successfully saved 48 abstracts for COVID-19.

--- Processing Disease: Monkeypox ---
Found 50 candidates for Monkeypox. Downloading details...
Successfully saved 48 abstracts for Monkeypox.

--- Processing Disease: Measles ---
Found 50 candidates for Measles. Downloading details...
Successfully saved 50 abstracts for Measles.

--- Processing Disease: Influenza ---
Found 50 candidates for Influenza. Downloading details...
Successfully saved 50 abstracts for Influenza.

--- Processing Disease: Ebola ---
Found 50 candidates for Ebola. Downloading details...
Successfully saved 50 abstracts for Ebola.


In [7]:
with open(OUTPUT_FILE, "w") as f:
        json.dump(data, f, indent=2)
        
print(f"\nSuccess! Saved {len(data)} abstracts to '{OUTPUT_FILE}'.")
print("You can now transfer this file to your GPU server.")


Success! Saved 246 abstracts to 'raw_abstracts.json'.
You can now transfer this file to your GPU server.
